## Nested Models - Follow-Along Practice

This notebook covers nested models and serialization in Pydantic.

#### Install Dependencies

Run the following cell to install Pydantic in your notebook environment.

In [ ]:
!pip install pydantic

#### 1. Models Inside Models

Use one Pydantic model as a field type in another model to form nested schemas.

In [ ]:
from pydantic import BaseModel

class OrderItem(BaseModel):
    product_id: str
    name: str
    quantity: int
    price: float

class Order(BaseModel):
    order_id: str
    item: OrderItem  # Nested model

# Create with nested data
order = Order(
    order_id="ORD-001",
    item=OrderItem(
        product_id="P1",
        name="Widget",
        quantity=2,
        price=29.99
    )
)

print("Item Name:", order.item.name)  # Widget

#### 2. Creating from Dictionaries

Pydantic automatically validates and instantiates nested models when parsed from nested dictionaries.

In [ ]:
data = {
    "order_id": "ORD-001",
    "item": {
        "product_id": "P1",
        "name": "Widget",
        "quantity": 2,
        "price": 29.99
    }
}

# Pydantic validates everything, including nested data
order = Order.model_validate(data)

print("Order ID:", order.order_id)     # ORD-001
print("Nested Item Name:", order.item.name)    # Widget

#### 3. Lists of Models

Handle collections of nested objects by specifying lists of nested models.

In [ ]:
class OrderList(BaseModel):
    order_id: str
    customer_email: str
    items: list[OrderItem]  # List of models

# From API response
order_data = {
    "order_id": "ORD-001",
    "customer_email": "customer@example.com",
    "items": [
        {"product_id": "P1", "name": "Widget", "quantity": 2, "price": 29.99},
        {"product_id": "P2", "name": "Gadget", "quantity": 1, "price": 49.99}
    ]
}

order = OrderList(**order_data)

print(f"Order {order.order_id}")
for item in order.items:
    print(f"  - {item.name}: {item.quantity} x ${item.price}")

#### 4. Optional Nested Models

Allow nested structures to be optional by using union types.

In [ ]:
class Discount(BaseModel):
    code: str
    percent: float

class OrderOptional(BaseModel):
    order_id: str
    total: float
    discount: Discount | None = None  # Optional nested model

# Works without discount
order1 = OrderOptional(order_id="ORD-001", total=99.99)
print("Discount is None:", order1.discount)  # None

# Works with discount
order2 = OrderOptional(
    order_id="ORD-002",
    total=99.99,
    discount={"code": "SAVE20", "percent": 20.0}
)
print("Discount Code:", order2.discount.code)  # SAVE20

#### 5. Deep Nesting

You can nest schemas as deep as needed to model complex objects.

In [ ]:
class Address(BaseModel):
    street: str
    city: str
    country: str

class Customer(BaseModel):
    name: str
    email: str
    billing_address: Address
    shipping_address: Address | None = None

class OrderDeep(BaseModel):
    order_id: str
    customer: Customer
    items: list[OrderItem]
    notes: str | None = None

data = {
    "order_id": "ORD-123",
    "customer": {
        "name": "Alice Smith",
        "email": "alice@example.com",
        "billing_address": {
            "street": "123 Main St",
            "city": "Amsterdam",
            "country": "Netherlands"
        }
    },
    "items": [
        {"product_id": "SKU-001", "name": "Widget", "quantity": 3, "price": 19.99}
    ]
}

order = OrderDeep.model_validate(data)
print("Billing City:", order.customer.billing_address.city)  # Amsterdam

#### 6. Converting Nested Models

When calling `model_dump()`, nested objects are automatically converted to standard python dictionaries as well.

In [ ]:
order = Order(
    order_id="ORD-001",
    item=OrderItem(name="Widget", price=29.99, product_id="P1", quantity=1)
)

# Converts everything to dict recursively
exported_data = order.model_dump()
print("Dumped dict:", exported_data)

#### 7. Common Patterns

#### Reusing Models Across Your Codebase

In [ ]:
class Money(BaseModel):
    amount: float
    currency: str = "USD"

class ProductMoney(BaseModel):
    name: str
    price: Money

class Invoice(BaseModel):
    items: list[ProductMoney]
    subtotal: Money
    tax: Money
    total: Money

# Reusing Money schema allows consistent structure across different parts of the code.

#### Self-Referencing Models (Trees)

In [ ]:
from __future__ import annotations

class Comment(BaseModel):
    text: str
    author: str
    replies: list[Comment] = []

comment = Comment(
    text="Great article!",
    author="Alice",
    replies=[
        Comment(text="Thanks!", author="Bob")
    ]
)
print("Root Comment:", comment.text)
print("Reply Comment:", comment.replies[0].text)